# COVID analysis — starter notebook

Works on `covid-combo.db` (SQLite). Tables:

| Table | Source | What |
|---|---|---|
| `covid_data` | OWID `owid-covid-data.csv` | cases, deaths, excess mortality, vaccination, demographics per country/day |
| `vaccinations_by_manufacturer` | OWID `vaccinations-by-manufacturer.csv` | cumulative doses per vaccine, 44 countries |
| `locations` | OWID `locations.csv` | which vaccines each country used (no counts) |
| `ve_primary_series` | IVAC VIEW-hub | vaccine-effectiveness estimates from published studies, primary series |
| `ve_booster` | IVAC VIEW-hub | same, for boosters |

Kernel: **syntra-ds**.

In [ ]:
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

conn = sqlite3.connect("covid-combo.db")


def sql(query: str, **kwargs) -> pd.DataFrame:
    """Run a query against covid-combo.db and return a DataFrame."""
    return pd.read_sql(query, conn, **kwargs)

In [ ]:
# Tables and row counts
tables = sql("SELECT name FROM sqlite_master WHERE type='table'").name
pd.DataFrame({
    "table": tables,
    "rows": [sql(f"SELECT COUNT(*) n FROM {t}").n[0] for t in tables],
})

In [ ]:
# Columns of the main table
sql("PRAGMA table_info(covid_data)")[["name", "type"]].T

## 1. One country over time

Change `COUNTRIES` to compare others.

In [ ]:
COUNTRIES = ("Belgium", "Denmark", "Sweden")

placeholders = ",".join("?" * len(COUNTRIES))
ts = sql(f"""
    SELECT location, date, new_deaths_smoothed_per_million,
           people_fully_vaccinated_per_hundred, excess_mortality
    FROM covid_data
    WHERE location IN ({placeholders})
""", params=COUNTRIES, parse_dates=["date"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for loc, g in ts.groupby("location"):
    ax1.plot(g.date, g.new_deaths_smoothed_per_million, label=loc)
    ax2.plot(g.date, g.people_fully_vaccinated_per_hundred, label=loc)
ax1.set_ylabel("deaths / million / day (7d avg)")
ax2.set_ylabel("% fully vaccinated")
ax1.legend()
plt.tight_layout()

## 2. Which countries dealt best with COVID?

Measure: **cumulative excess mortality per million, 2020 → end 2022** — counts all extra deaths,
so it isn't fooled by countries that tested or reported poorly.

Filters: population ≥ 1M, last excess-mortality value between 2022-10-01 and 2022-12-31.
Coverage is mostly richer/middle-income countries (Africa, India, China largely missing).

In [ ]:
CUTOFF = "2022-12-31"
MIN_POP = 1_000_000

ranking = sql("""
WITH em AS (
  SELECT location, date, excess_mortality_cumulative_per_million AS excess_pm,
         ROW_NUMBER() OVER (PARTITION BY location ORDER BY date DESC) AS rn
  FROM covid_data
  WHERE excess_mortality_cumulative_per_million IS NOT NULL
    AND iso_code NOT LIKE 'OWID_%' AND date <= :cutoff),
rep AS (
  SELECT location, MAX(total_deaths_per_million) AS reported_pm
  FROM covid_data WHERE date <= :cutoff GROUP BY location),
st AS (
  SELECT location, MAX(continent) AS continent, MAX(population) AS population,
         MAX(aged_65_older) AS pct_65plus, MAX(gdp_per_capita) AS gdp_per_capita,
         MAX(people_fully_vaccinated_per_hundred) AS pct_fully_vax,
         AVG(stringency_index) AS avg_stringency
  FROM covid_data WHERE date <= :cutoff GROUP BY location)
SELECT em.location, st.continent, date(em.date) AS last_date,
       em.excess_pm, rep.reported_pm, st.population, st.pct_65plus,
       st.gdp_per_capita, st.pct_fully_vax, st.avg_stringency
FROM em JOIN rep USING (location) JOIN st USING (location)
WHERE em.rn = 1 AND em.date >= date(:cutoff, '-3 months') AND st.population >= :min_pop
ORDER BY em.excess_pm
""", params={"cutoff": CUTOFF, "min_pop": MIN_POP})

ranking["excess_vs_reported"] = ranking.excess_pm / ranking.reported_pm
print(f"{len(ranking)} countries")
ranking.head(15).round(1)

In [ ]:
# Worst 15
ranking.tail(15).iloc[::-1].round(1)

In [ ]:
# What goes together with low excess mortality? (correlation, not causation)
ranking[["excess_pm", "pct_fully_vax", "gdp_per_capita", "pct_65plus", "avg_stringency"]] \
    .corr().loc["excess_pm"].round(2)

In [ ]:
# Excess mortality vs. vaccination rate
fig, ax = plt.subplots(figsize=(10, 6))
for cont, g in ranking.groupby("continent"):
    ax.scatter(g.pct_fully_vax, g.excess_pm, label=cont, alpha=0.75)
for _, r in ranking.iterrows():
    ax.annotate(r.location, (r.pct_fully_vax, r.excess_pm), fontsize=7, alpha=0.7)
ax.set_xlabel("% fully vaccinated (max by end 2022)")
ax.set_ylabel("cumulative excess deaths per million (2020–2022)")
ax.legend()
plt.tight_layout()

### Rough age adjustment

Older populations are expected to have more deaths. A simple linear fit of excess mortality on % aged 65+
gives an "expected" value; `vs_expected < 0` means better than expected.
This is a crude correction (the fit explains little) — proper age-standardised data would be fairer.

In [ ]:
m = ranking.dropna(subset=["pct_65plus"])
slope, intercept = np.polyfit(m.pct_65plus, m.excess_pm, 1)
ranking["vs_expected"] = ranking.excess_pm - (intercept + slope * ranking.pct_65plus)
print(f"expected excess_pm = {intercept:.0f} + {slope:.0f} × %65+   "
      f"(corr = {np.corrcoef(m.pct_65plus, m.excess_pm)[0, 1]:.2f})")
ranking.sort_values("vs_expected")[["location", "excess_pm", "pct_65plus", "vs_expected"]].head(12).round(0)

## 3. Which vaccine worked best? (VIEW-hub)

Each row in `ve_primary_series` is one VE estimate (%) with a 95% CI from one study.
We look at **death**, **completed primary series** (`dose = 'final'`), and split by variant period,
because Omicron lowered VE for every vaccine and different vaccines were studied in different periods.

In [ ]:
ve = sql("""
    SELECT study_id, first_author, primary_series_vaccine AS vaccine, country,
           population_category, variant_context, ve, lcl, ucl
    FROM ve_primary_series
    WHERE outcome_category = 'death' AND dose = 'final'
      AND primary_series_vaccine NOT LIKE '% vs %'   -- relative comparisons, not VE vs. unvaccinated
      AND ve IS NOT NULL
""")


def variant_period(v) -> str:
    if v is None or pd.isna(v):
        return "unknown"
    v = v.lower()
    if v.startswith("omicron"):
        return "omicron"
    return "mixed" if "omicron" in v else "pre-omicron"


ve["period"] = ve.variant_context.map(variant_period)
ve.period.value_counts()

In [ ]:
# Crude summary: plain average of all estimates
(ve.groupby(["period", "vaccine"])
   .agg(estimates=("ve", "size"), studies=("study_id", "nunique"),
        countries=("country", "nunique"), avg_ve=("ve", "mean"))
   .query("studies >= 3")
   .round(1)
   .sort_values(["period", "avg_ve"], ascending=[True, False]))

### Weighted (inverse-variance) pooled VE

Standard meta-analysis approach: work on the log relative-risk scale, `log(1 − VE)`,
derive the standard error from the CI, and weight each estimate by `1 / SE²`
so precise studies count more. This is a simple **fixed-effect** pool that treats estimates as independent —
several come from the same study, so the CIs are too narrow. Good for a ranking, not for publication.

In [ ]:
def pooled_ve(df: pd.DataFrame) -> pd.Series:
    d = df.dropna(subset=["ve", "lcl", "ucl"])
    d = d[(d.ve < 100) & (d.ucl < 100) & (d.lcl < d.ucl)]
    log_rr = np.log(1 - d.ve / 100)
    se = (np.log(1 - d.lcl / 100) - np.log(1 - d.ucl / 100)) / (2 * 1.96)
    w = 1 / se**2
    est = (w * log_rr).sum() / w.sum()
    se_pooled = np.sqrt(1 / w.sum())
    return pd.Series({
        "estimates": len(d),
        "studies": d.study_id.nunique(),
        "pooled_ve": 100 * (1 - np.exp(est)),
        "ci_low": 100 * (1 - np.exp(est + 1.96 * se_pooled)),
        "ci_high": 100 * (1 - np.exp(est - 1.96 * se_pooled)),
    })


pooled = (ve.groupby(["period", "vaccine"])[["study_id", "ve", "lcl", "ucl"]]
            .apply(pooled_ve)
            .query("studies >= 3")
            .round(1)
            .sort_values(["period", "pooled_ve"], ascending=[True, False]))
pooled

In [ ]:
# Forest-style plot per period
periods = [p for p in ("pre-omicron", "omicron") if p in pooled.index.get_level_values(0)]
fig, axes = plt.subplots(1, len(periods), figsize=(13, 5), sharex=True)
for ax, p in zip(np.atleast_1d(axes), periods):
    d = pooled.loc[p].sort_values("pooled_ve")
    ax.errorbar(d.pooled_ve, range(len(d)),
                xerr=[d.pooled_ve - d.ci_low, d.ci_high - d.pooled_ve], fmt="o", capsize=3)
    ax.set_yticks(range(len(d)), [f"{v}  (n={int(s)})" for v, s in zip(d.index, d.studies)], fontsize=8)
    ax.set_title(f"VE against death — {p}")
    ax.set_xlabel("pooled VE (%)")
    ax.axvline(0, color="grey", lw=0.5)
plt.tight_layout()

In [ ]:
# Drill down: all death estimates for one vaccine
VACCINE = "CoronaVac (Sinovac)"
ve[ve.vaccine == VACCINE].sort_values("ve")

## 4. Vaccine mix per country (OWID manufacturer data, 44 countries)

In [ ]:
COUNTRY = "Belgium"
mix = sql("""
    SELECT date, vaccine, total_vaccinations
    FROM vaccinations_by_manufacturer WHERE location = ?
""", params=(COUNTRY,), parse_dates=["date"])

mix.pivot_table(index="date", columns="vaccine", values="total_vaccinations") \
   .ffill().plot(figsize=(11, 5), title=f"Cumulative doses by vaccine — {COUNTRY}")
plt.ylabel("doses")
plt.tight_layout()

In [ ]:
# Final share of doses per vaccine, per country
latest = sql("""
    SELECT location, vaccine, MAX(total_vaccinations) AS doses
    FROM vaccinations_by_manufacturer GROUP BY location, vaccine
""")
share = latest.pivot_table(index="location", columns="vaccine", values="doses", fill_value=0)
(share.div(share.sum(axis=1), axis=0) * 100).round(1)

## 5. Which vaccines did each country use? (`locations`, all countries, no counts)

In [ ]:
locs = sql("SELECT location, iso_code, vaccines FROM locations")
long = locs.assign(vaccine=locs.vaccines.str.split(",\\s*")).explode("vaccine")
long.vaccine.value_counts().head(15)

## 6. Who coped best? Deaths vs. economic cost

Three dimensions per country:

- **y — health:** cumulative excess deaths per million up to `EM_CUTOFF` (end 2023 by default, so
  countries that only *postponed* deaths — e.g. NZ, Australia — are judged fairly)
- **x — economy:** cumulative GDP loss 2020–2023 vs. the 2015–2019 trend (`gdp_covid_loss`)
- **colour — restrictions:** average `stringency_index` 2020–2022 (how much lockdown it took)

**Bottom-left = few deaths at low economic cost.** Dashed lines are the medians of the plotted group.
GDP loss also contains non-COVID shocks (energy crisis 2022, wars, oil booms) — use `PEER_GROUP`
to compare similar countries, and `MAX_ABS_GDP_LOSS` to drop extreme outliers.

In [ ]:
EM_CUTOFF = "2023-12-31"        # excess mortality measured up to this date
EM_MAX_STALENESS = "-6 months"  # last excess-mortality value must be within this window of EM_CUTOFF
MIN_POP = 1_000_000
MAX_ABS_GDP_LOSS = 25           # drop countries whose GDP "loss" is dominated by other shocks
PEER_GROUP = None               # e.g. ["Europe"] (continents) or None for all

coped = sql("""
WITH em AS (
  SELECT location, iso_code, date, excess_mortality_cumulative_per_million AS excess_pm,
         ROW_NUMBER() OVER (PARTITION BY location ORDER BY date DESC) AS rn
  FROM covid_data
  WHERE excess_mortality_cumulative_per_million IS NOT NULL
    AND iso_code NOT LIKE 'OWID_%' AND date <= :cutoff),
st AS (
  SELECT location, MAX(continent) AS continent, MAX(population) AS population,
         MAX(aged_65_older) AS pct_65plus,
         MAX(people_fully_vaccinated_per_hundred) AS pct_fully_vax,
         AVG(CASE WHEN date < '2023-01-01' THEN stringency_index END) AS avg_stringency
  FROM covid_data GROUP BY location)
SELECT em.location, em.iso_code, st.continent, date(em.date) AS em_last_date, em.excess_pm,
       g.cum_loss_2020_2023_pct_of_trend AS gdp_loss_pct, g.growth_2020_pct,
       st.avg_stringency, st.pct_fully_vax, st.pct_65plus, st.population
FROM em
JOIN st USING (location)
JOIN gdp_covid_loss g ON g.iso_code = em.iso_code
WHERE em.rn = 1
  AND em.date >= date(:cutoff, :stale)
  AND st.population >= :min_pop
""", params={"cutoff": EM_CUTOFF, "stale": EM_MAX_STALENESS, "min_pop": MIN_POP})

dropped = coped[coped.gdp_loss_pct.abs() > MAX_ABS_GDP_LOSS].location.tolist()
coped = coped[coped.gdp_loss_pct.abs() <= MAX_ABS_GDP_LOSS]
if PEER_GROUP:
    coped = coped[coped.continent.isin(PEER_GROUP)]
print(f"{len(coped)} countries plotted; dropped as GDP outliers: {dropped or 'none'}")
coped.sort_values("excess_pm").head(10).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
size = np.sqrt(coped.population) / 60 + 15  # dot size ~ population
has_s = coped.avg_stringency.notna()
sc = ax.scatter(coped.gdp_loss_pct[has_s], coped.excess_pm[has_s], c=coped.avg_stringency[has_s],
                cmap="viridis_r", s=size[has_s], alpha=0.85, edgecolors="grey", linewidths=0.5)
ax.scatter(coped.gdp_loss_pct[~has_s], coped.excess_pm[~has_s], s=size[~has_s],
           facecolors="none", edgecolors="grey", label="no stringency data")
if (~has_s).any():
    ax.legend(loc="upper left", fontsize=8)
for _, r in coped.iterrows():
    ax.annotate(r.location, (r.gdp_loss_pct, r.excess_pm), fontsize=7, alpha=0.8,
                xytext=(3, 3), textcoords="offset points")

ax.axvline(coped.gdp_loss_pct.median(), color="grey", ls="--", lw=0.8)
ax.axhline(coped.excess_pm.median(), color="grey", ls="--", lw=0.8)
ax.text(0.01, 0.01, "few deaths, low economic cost", transform=ax.transAxes, color="green", fontsize=9)
ax.text(0.99, 0.99, "many deaths, high economic cost", transform=ax.transAxes, color="firebrick",
        fontsize=9, ha="right", va="top")

ax.set_xlabel("cumulative GDP loss 2020–2023 vs. pre-COVID trend (% of trend GDP)")
ax.set_ylabel(f"cumulative excess deaths per million (2020 → {EM_CUTOFF[:4]})")
ax.set_title("Who coped best? Excess deaths vs. economic cost (colour = avg. stringency 2020–22)")
fig.colorbar(sc, ax=ax, label="average stringency index (0–100)")
plt.tight_layout()

### Combined ranking (equal weights)

Rank each country on excess deaths and on GDP loss (1 = best), then average the two ranks.
The weights are a choice — change `W_HEALTH` and check whether the top of the list is stable.

In [ ]:
W_HEALTH = 0.5  # 0.5 = health and economy count equally

ranked = coped.assign(
    rank_health=coped.excess_pm.rank(),
    rank_economy=coped.gdp_loss_pct.rank(),
)
ranked["combined_score"] = W_HEALTH * ranked.rank_health + (1 - W_HEALTH) * ranked.rank_economy
ranked = ranked.sort_values("combined_score")
cols = ["location", "continent", "excess_pm", "gdp_loss_pct", "avg_stringency", "pct_fully_vax",
        "rank_health", "rank_economy", "combined_score"]
ranked[cols].head(15).round(1)

In [ ]:
# Worst 10 on the combined score
ranked[cols].tail(10).iloc[::-1].round(1)

In [ ]:
conn.close()